# JEPA-for-Trading V4: risk-off world model planner

V4 fixes the main V3 limitation: V3 found a good allocation, then mostly held it. V4 makes risk-off behavior explicit.

Changes:
- utility labels penalize future drawdown, volatility, turnover and costs more strongly;
- samples with bad future `hold` outcomes boost `cash/derisk`;
- planner uses a transparent outcome score, not only learned energy;
- hard risk-off rule can override `hold`;
- backtest saves per-asset weights and full planner diagnostics to `outputs/v4_latest/`.


In [ ]:
# Kaggle/bootstrap cell. Run this first.
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/aurvl/jepa-for-trading.git"
BRANCH = "version4"
REPO_DIR = Path("/kaggle/working/jepa-for-trading") if Path("/kaggle/working").exists() else Path.cwd()

if not (REPO_DIR / "src" / "jepa_trading").exists():
    if REPO_DIR.exists():
        print(f"Repo dir exists but package not found: {REPO_DIR}")
    else:
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())
!{sys.executable} -m pip install -q -e . --no-deps

src_path = str(REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)


In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from jepa_trading.config import ensure_dirs, load_config
from jepa_trading.data.pipeline import prepare_market_data, create_v4_dataloaders
from jepa_trading.data.v2_dataset import PortfolioActionConfig
from jepa_trading.data.v4_dataset import summarize_v4_batch
from jepa_trading.evaluation.backtest import (
    buy_and_hold_weight,
    equal_weight,
    momentum_weight,
    random_long_only_weight,
    run_weight_strategy,
    volatility_target_weight,
)
from jepa_trading.evaluation.metrics import metrics_table, validate_backtest_histories
from jepa_trading.evaluation.plots import plot_drawdown, plot_equity_curves, plot_turnover
from jepa_trading.evaluation.statistical_tests import bootstrap_mean_return_p_value, randomization_p_value
from jepa_trading.evaluation.v4_backtest import run_v4_planner_backtest
from jepa_trading.models.world_model_v2 import V2WorldModel
from jepa_trading.planning.v4_planner import V4RiskOffPlanner
from jepa_trading.rl.env import TradingEnv
from jepa_trading.rl.observer import RawMarketObserver
from jepa_trading.training.checkpoints import load_checkpoint
from jepa_trading.training.train_v4 import train_v4_world_model
from jepa_trading.utils.device import get_device
from jepa_trading.utils.seed import seed_everything


In [ ]:
config = load_config("configs/default.yaml")

macro_candidates = []
if Path("/kaggle/input").exists():
    macro_candidates += list(Path("/kaggle/input").rglob("macro_data.parquet"))
    macro_candidates += list(Path("/kaggle/input").rglob("estimated_volatility_with_macro.csv"))
if macro_candidates:
    config["data"]["macro_path"] = str(macro_candidates[0])
    print("Using macro file:", config["data"]["macro_path"])
else:
    print("Using configured macro file:", config["data"]["macro_path"])

FAST_DEV_RUN = False
if FAST_DEV_RUN:
    config["data"]["tickers"] = config["data"]["tickers"][:8]
    config["v4"]["world_max_steps"] = 50
    config["v4"]["world_warmup_steps"] = 10
    config["v4"]["planner_sampled_actions"] = 16

ensure_dirs(config)
seed_everything(config["seed"])
device = get_device(config["device"])
device


In [ ]:
prepared_df, arrays, feature_columns = prepare_market_data(config, force_download=False)
loaders = create_v4_dataloaders(config, arrays)

print("rows:", len(prepared_df))
print("assets:", len(arrays.tickers), list(arrays.tickers))
print("features:", len(feature_columns), feature_columns)
print("dates:", arrays.dates.min(), "->", arrays.dates.max())
print("dataset sizes:", {k: len(v.dataset) for k, v in loaders.items()})

batch = next(iter(loaders["train"]))
print("batch summary:", summarize_v4_batch(batch))


In [ ]:
portfolio_cfg = config["portfolio"]
v4_cfg = config["v4"]

model = V2WorldModel(
    n_features=len(feature_columns),
    max_assets=len(arrays.tickers),
    portfolio_state_dim=batch["portfolio_state"].shape[-1],
    action_dim=batch["actions"].shape[-1],
    d_model=config["model"]["d_model"],
    latent_dim=config["model"]["latent_dim"],
    n_heads=config["model"]["n_heads"],
    n_layers=config["model"]["n_layers"],
    dropout=config["model"]["dropout"],
    ema_decay=config["training"]["ema_decay"],
    hidden_dim=config["v2"].get("hidden_dim", 256),
    policy_mode=portfolio_cfg.get("mode", "long_only"),
    max_abs_weight=portfolio_cfg.get("max_long_weight", portfolio_cfg.get("max_weight_per_asset", 0.15)),
    asset_embedding_dim=config["model"].get("asset_embedding_dim"),
)

v4_ckpt = Path(config["training"]["checkpoint_dir"]) / "v4_world_model.pt"
if v4_ckpt.exists():
    print("Loading existing V4 checkpoint:", v4_ckpt)
    metadata = load_checkpoint(v4_ckpt, model, map_location=device)
    print("checkpoint metadata:", metadata)
else:
    v4_history = train_v4_world_model(
        model=model,
        train_loader=loaders["train"],
        val_loader=loaders["val"],
        device=device,
        max_steps=v4_cfg["world_max_steps"],
        warmup_steps=v4_cfg["world_warmup_steps"],
        lr=config["training"]["lr"],
        weight_decay=config["training"]["weight_decay"],
        checkpoint_path=v4_ckpt,
        weights=v4_cfg["loss_weights"],
        rank_margin=v4_cfg["rank_margin"],
        eval_every=config["training"]["eval_every"],
        log_every=config["training"]["log_every"],
    )
    display(v4_history.tail())
    load_checkpoint(v4_ckpt, model, map_location=device)

model.to(device).eval()
print("model ready")


In [ ]:
# Local imports + forced reload make this cell robust after Kaggle kernel restarts, pulls, or partial reruns.
import importlib
import inspect
import jepa_trading.planning.v4_planner as v4_planner_module
import jepa_trading.evaluation.v4_backtest as v4_backtest_module
importlib.reload(v4_planner_module)
importlib.reload(v4_backtest_module)

from jepa_trading.data.v2_dataset import PortfolioActionConfig
from jepa_trading.planning.v4_planner import V4RiskOffPlanner
from jepa_trading.evaluation.v4_backtest import run_v4_planner_backtest
from jepa_trading.rl.env import TradingEnv
from jepa_trading.rl.observer import RawMarketObserver

print("Loaded V4RiskOffPlanner", getattr(V4RiskOffPlanner, "CODE_VERSION", "missing-code-version"))
assert "_sanitize_outcomes" in inspect.getsource(V4RiskOffPlanner), "Old V4 planner loaded. Pull latest version4 and rerun this cell."

action_cfg = PortfolioActionConfig(
    mode=portfolio_cfg.get("mode", "long_only"),
    cash_initial=portfolio_cfg["cash_initial"],
    transaction_cost_bps=portfolio_cfg["transaction_cost_bps"],
    max_long_weight=portfolio_cfg.get("max_long_weight", portfolio_cfg.get("max_weight_per_asset", 0.15)),
    max_short_weight=portfolio_cfg.get("max_short_weight", 0.05),
    max_gross_exposure=portfolio_cfg.get("max_gross_exposure", 1.0),
    max_net_exposure=portfolio_cfg.get("max_net_exposure", 1.0),
    borrow_cost_bps=portfolio_cfg.get("borrow_cost_bps", 2.0),
    n_action_samples=v4_cfg["planner_sampled_actions"],
    derisk_fraction=portfolio_cfg.get("derisk_fraction", 0.50),
)
planner_cfg = v4_cfg["planner"]
planner = V4RiskOffPlanner(
    model=model,
    action_config=action_cfg,
    horizons=config["data"]["horizons"],
    n_sampled_actions=v4_cfg["planner_sampled_actions"],
    model_score_weight=planner_cfg["model_score_weight"],
    return_weight=planner_cfg["return_weight"],
    drawdown_penalty=planner_cfg["drawdown_penalty"],
    volatility_penalty=planner_cfg["volatility_penalty"],
    turnover_penalty=planner_cfg["turnover_penalty"],
    hard_risk_off_drawdown=planner_cfg["hard_risk_off_drawdown"],
    hard_risk_off_return=planner_cfg["hard_risk_off_return"],
    min_action_advantage=planner_cfg["min_action_advantage"],
    cash_bootstrap_min_return=planner_cfg.get("cash_bootstrap_min_return", -0.03),
    cash_bootstrap_max_drawdown=planner_cfg.get("cash_bootstrap_max_drawdown", -0.12),
    cash_bootstrap_score_tolerance=planner_cfg.get("cash_bootstrap_score_tolerance", 0.15),
    rebalance_every=planner_cfg["rebalance_every"],
    force_recheck_after=planner_cfg["force_recheck_after"],
    chunk_size=256,
    device=device,
    seed=config["seed"] + 400,
)

start_date = pd.Timestamp(config["data"]["val_end"]) + pd.offsets.BDay(1)
end_date = pd.Timestamp(arrays.dates[-2])
common_env_kwargs = dict(
    arrays=arrays,
    observer=RawMarketObserver(arrays, config["data"]["lookback"]),
    start_date=start_date,
    end_date=end_date,
    lookback=config["data"]["lookback"],
    cash_initial=portfolio_cfg["cash_initial"],
    transaction_cost_bps=portfolio_cfg["transaction_cost_bps"],
    max_weight_per_asset=portfolio_cfg.get("max_weight_per_asset", portfolio_cfg.get("max_long_weight", 0.15)),
    max_turnover=portfolio_cfg["max_turnover"],
    mode=portfolio_cfg.get("mode", "long_only"),
    max_long_weight=portfolio_cfg.get("max_long_weight", portfolio_cfg.get("max_weight_per_asset", 0.15)),
    max_short_weight=portfolio_cfg.get("max_short_weight", 0.05),
    max_gross_exposure=portfolio_cfg.get("max_gross_exposure", 1.0),
    max_net_exposure=portfolio_cfg.get("max_net_exposure", 1.0),
    borrow_cost_bps=portfolio_cfg.get("borrow_cost_bps", 2.0),
)

agent_history = run_v4_planner_backtest(
    arrays=arrays,
    planner=planner,
    start_date=start_date,
    end_date=end_date,
    lookback=config["data"]["lookback"],
    cash_initial=portfolio_cfg["cash_initial"],
    transaction_cost_bps=portfolio_cfg["transaction_cost_bps"],
    max_weight_per_asset=portfolio_cfg.get("max_weight_per_asset", portfolio_cfg.get("max_long_weight", 0.15)),
    max_turnover=portfolio_cfg["max_turnover"],
    mode=portfolio_cfg.get("mode", "long_only"),
    max_long_weight=portfolio_cfg.get("max_long_weight", portfolio_cfg.get("max_weight_per_asset", 0.15)),
    max_short_weight=portfolio_cfg.get("max_short_weight", 0.05),
    max_gross_exposure=portfolio_cfg.get("max_gross_exposure", 1.0),
    max_net_exposure=portfolio_cfg.get("max_net_exposure", 1.0),
    borrow_cost_bps=portfolio_cfg.get("borrow_cost_bps", 2.0),
)

print(agent_history.tail())
print(agent_history["selected_action_name"].value_counts(normalize=True))
print(agent_history["selected_reason"].value_counts(normalize=True))


In [ ]:
def make_env():
    return TradingEnv(**common_env_kwargs)

buy_hold = run_weight_strategy(make_env(), buy_and_hold_weight)
equal_hist = run_weight_strategy(make_env(), equal_weight)
momentum_hist = run_weight_strategy(make_env(), lambda env, mask: momentum_weight(env, mask, lookback=20))
vol_target_hist = run_weight_strategy(make_env(), volatility_target_weight)

N_RANDOM = 100
random_histories = []
for i in range(N_RANDOM):
    rng = np.random.default_rng(config["seed"] + 2000 + i)
    random_histories.append(run_weight_strategy(make_env(), lambda env, mask, rng=rng: random_long_only_weight(env, mask, rng)))

histories = {
    "V4 JEPA Risk-Off Planner": agent_history,
    "Buy & Hold": buy_hold,
    "Equal Weight": equal_hist,
    "Momentum": momentum_hist,
    "Vol Target": vol_target_hist,
}
validity = validate_backtest_histories(histories)
display(validity)
metrics = metrics_table(histories)
display(metrics)

if not validity["valid_backtest"].all():
    print("INVALID BACKTEST: do not interpret performance positively.")
else:
    random_p = randomization_p_value(agent_history, random_histories)
    boot_vs_bh = bootstrap_mean_return_p_value(agent_history, buy_hold)
    print("p_value_random_beats_agent:", random_p)
    print("bootstrap agent minus buy_hold:", boot_vs_bh)


In [ ]:
plot_equity_curves(
    agent_history,
    buy_hold,
    random_histories,
    extra={"Equal Weight": equal_hist, "Momentum": momentum_hist, "Vol Target": vol_target_hist},
)
plot_drawdown(agent_history, label="V4 JEPA Risk-Off Planner")
plot_turnover(agent_history)

fig, ax = plt.subplots(figsize=(14, 4))
agent_history["selected_action_name"].value_counts().plot(kind="bar", ax=ax, color="#39ff14")
ax.set_title("V4 selected action types")
ax.grid(alpha=0.25, axis="y")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(14, 4))
agent_history["selected_reason"].value_counts().plot(kind="bar", ax=ax, color="#39ff14")
ax.set_title("V4 selected reasons")
ax.grid(alpha=0.25, axis="y")
plt.tight_layout()
plt.show()


In [ ]:
# Save all V4 run outputs for post-run diagnosis.
OUTPUT_DIR = Path("outputs/v4_latest")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

agent_history.to_csv(OUTPUT_DIR / "agent_history.csv", index=False)
buy_hold.to_csv(OUTPUT_DIR / "buy_hold_history.csv", index=False)
equal_hist.to_csv(OUTPUT_DIR / "equal_weight_history.csv", index=False)
momentum_hist.to_csv(OUTPUT_DIR / "momentum_history.csv", index=False)
vol_target_hist.to_csv(OUTPUT_DIR / "vol_target_history.csv", index=False)

weight_cols = [c for c in agent_history.columns if c.startswith("weight_")]
agent_history[["date", "equity", "cash_weight", "gross_exposure", *weight_cols]].to_csv(OUTPUT_DIR / "portfolio_weights.csv", index=False)

random_rows = []
for i, hist in enumerate(random_histories):
    hist.to_csv(OUTPUT_DIR / f"random_history_{i:03d}.csv", index=False)
    random_rows.append({
        "random_id": i,
        "start_equity": float(hist["equity"].iloc[0]),
        "final_equity": float(hist["equity"].iloc[-1]),
        "total_return": float(hist["equity"].iloc[-1] / hist["equity"].iloc[0] - 1.0),
        "avg_turnover": float(hist.get("turnover", pd.Series(0.0, index=hist.index)).mean()),
    })
pd.DataFrame(random_rows).to_csv(OUTPUT_DIR / "random_summary.csv", index=False)
metrics.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
validity.to_csv(OUTPUT_DIR / "validity.csv", index=False)
agent_history["selected_action_name"].value_counts().rename_axis("action").reset_index(name="count").to_csv(OUTPUT_DIR / "action_counts.csv", index=False)
agent_history["selected_reason"].value_counts().rename_axis("reason").reset_index(name="count").to_csv(OUTPUT_DIR / "selection_reason_counts.csv", index=False)

planner_cols = [
    "date", "equity", "cash_weight", "gross_exposure", "turnover", "days_since_trade",
    "selected_action_name", "selected_reason", "planned_horizon", "planner_score", "planner_raw_energy",
    "best_candidate_name", "best_candidate_score", "hold_score", "cash_score", "derisk_score",
    "best_risk_name", "best_risk_score", "hard_risk_off_flag",
    "predicted_log_return", "predicted_drawdown", "predicted_vol",
    "hold_predicted_log_return", "hold_predicted_drawdown", "hold_predicted_vol",
    "best_risk_predicted_log_return", "best_risk_predicted_drawdown", "best_risk_predicted_vol",
]
agent_history[[c for c in planner_cols if c in agent_history.columns]].to_csv(OUTPUT_DIR / "planner_diagnostics.csv", index=False)

summary = {
    "branch": "version4",
    "planner_code_version": getattr(V4RiskOffPlanner, "CODE_VERSION", "missing-code-version"),
    "checkpoint": str(v4_ckpt),
    "n_assets": int(len(arrays.tickers)),
    "n_features": int(len(feature_columns)),
    "test_start": str(start_date),
    "test_end": str(end_date),
    "v4_config": config["v4"],
    "portfolio_config": config["portfolio"],
    "action_counts": agent_history["selected_action_name"].value_counts().to_dict(),
    "selection_reason_counts": agent_history["selected_reason"].value_counts().to_dict(),
    "avg_cash_weight": float(agent_history["cash_weight"].mean()),
    "avg_gross_exposure": float(agent_history["gross_exposure"].mean()),
    "avg_turnover": float(agent_history["turnover"].mean()),
    "final_equity": float(agent_history["equity"].iloc[-1]),
}
(OUTPUT_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")

plt.style.use("dark_background")
fig, ax = plt.subplots(figsize=(16, 7))
for hist in random_histories:
    ax.plot(hist["date"], hist["equity"], color="gray", alpha=0.16, linewidth=0.8)
ax.plot(equal_hist["date"], equal_hist["equity"], linewidth=1.5, label="Equal Weight")
ax.plot(momentum_hist["date"], momentum_hist["equity"], linewidth=1.5, label="Momentum")
ax.plot(vol_target_hist["date"], vol_target_hist["equity"], linewidth=1.5, label="Vol Target")
ax.plot(buy_hold["date"], buy_hold["equity"], color="white", linewidth=2.0, label="Buy & Hold")
ax.plot(agent_history["date"], agent_history["equity"], color="#39ff14", linewidth=2.5, label="V4 JEPA Planner")
ax.set_title("Trading Strategy Equity Curves")
ax.set_xlabel("Date")
ax.set_ylabel("Equity")
ax.grid(alpha=0.18)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "equity_curves.png", dpi=160, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(14, 4))
dd = agent_history["equity"] / agent_history["equity"].cummax() - 1.0
ax.fill_between(agent_history["date"], dd, 0, color="#39ff14", alpha=0.35)
ax.plot(agent_history["date"], dd, color="#39ff14", label="V4 JEPA Planner")
ax.set_title("V4 Drawdown")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "drawdown.png", dpi=160, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(agent_history["date"], agent_history["cash_weight"], color="#39ff14")
ax.set_title("V4 Cash Weight")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "cash_weight.png", dpi=160, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(16, 6))
weights_for_plot = agent_history[["date", *weight_cols]].copy()
weights_for_plot = weights_for_plot.set_index("date")
weights_for_plot.iloc[:, : min(12, len(weight_cols))].plot(ax=ax, linewidth=1.0)
ax.set_title("Top logged asset weights")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "portfolio_weights.png", dpi=160, bbox_inches="tight")
plt.close(fig)

print("Saved V4 outputs to", OUTPUT_DIR.resolve())
for path in sorted(OUTPUT_DIR.glob("*"))[:40]:
    print("-", path)


In [ ]:
# Final cell: push the run artifacts/results back to GitHub branch version4.
# Requires a Kaggle Secret named GITHUB_TOKEN with repo write access.
import os
import subprocess
from pathlib import Path

BRANCH = "version4"
REMOTE_REPO = "github.com/aurvl/jepa-for-trading.git"
GIT_USER_NAME = "aurvl"
GIT_USER_EMAIL = "aurelvhei@outlook.fr"
COMMIT_MESSAGE = "Add V4 Kaggle run artifacts"

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = os.environ.get("GITHUB_TOKEN", "")

if not token:
    raise RuntimeError("Missing GITHUB_TOKEN. Add it in Kaggle Secrets before running this cell.")

def run(cmd, check=True):
    printable = " ".join(cmd)
    if token:
        printable = printable.replace(token, "TOKEN_REDACTED")
    print("$", printable)
    proc = subprocess.run(cmd, text=True, capture_output=True)
    out = (proc.stdout or "").replace(token, "TOKEN_REDACTED")
    err = (proc.stderr or "").replace(token, "TOKEN_REDACTED")
    if out.strip():
        print(out)
    if err.strip():
        print(err)
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=out, stderr=err)
    return proc

run(["git", "config", "user.name", GIT_USER_NAME])
run(["git", "config", "user.email", GIT_USER_EMAIL])
run(["git", "checkout", BRANCH], check=False)
run(["git", "lfs", "install"], check=False)
run(["git", "lfs", "track", "models/*.pt"], check=False)
run(["git", "lfs", "track", "outputs/**"], check=False)
run(["git", "add", "."])
for path in ["models", "outputs", "logs"]:
    if Path(path).exists():
        run(["git", "add", "-f", path], check=False)
status = run(["git", "status", "--short"], check=False)
if status.stdout.strip():
    run(["git", "commit", "-m", COMMIT_MESSAGE], check=False)
else:
    print("Nothing to commit; pushing current HEAD.")
auth_prefix = "https://" + "x-access-token" + ":"
auth_remote = f"{auth_prefix}{token}@{REMOTE_REPO}"
run(["git", "remote", "set-url", "origin", auth_remote])
run(["git", "pull", "--rebase", "origin", BRANCH], check=False)
run(["git", "push", "origin", f"HEAD:{BRANCH}"])
run(["git", "remote", "set-url", "origin", f"https://{REMOTE_REPO}"], check=False)
print("Pushed to", BRANCH)
